In [ ]:
import matplotlib.pyplot as pyplot
import pandas as pd
import geopandas as gpd
import numpy as np
from google.colab import drive

In [ ]:
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Nino/raw_data/era5_county_monthly.csv")
df["GEOID"] = df["GEOID"].astype(str).str.zfill(5)
df

,GEOID,date,volumetric_soil_water_layer_3,total_precipitation_sum
0,29510,1950-02,0.404795,0.097962
1,32510,1950-02,0.410405,0.053591
2,51580,1950-02,0.406927,0.085883
3,51530,1950-02,0.487131,0.075526
4,51678,1950-02,0.483780,0.074520
...,...,...,...,...
2850031,44003,2026-06,0.185105,0.053278
2850032,09007,2026-06,0.202359,0.066271
2850033,09013,2026-06,0.152113,0.071856
2850034,09003,2026-06,0.235094,0.083719


In [ ]:
# county 25019, nantucket returns na because it's small and was not assigned a landera5 value

df.sort_values(by="total_precipitation_sum")
df["total_precipitation_sum"].isna().sum()
na_precip_rows = df[df["total_precipitation_sum"].isna()]
print(na_precip_rows["GEOID"].nunique(), "counties affected")
print(na_precip_rows["GEOID"].value_counts().head(20))   # same counties repeatedly?
print(na_precip_rows["date"].value_counts().head(20))

1 counties affected
GEOID
25019    917
Name: count, dtype: int64
date
2026-06    1
1950-02    1
1950-03    1
1950-04    1
1950-05    1
2025-02    1
2025-01    1
2024-12    1
2024-11    1
2024-10    1
2024-09    1
2024-08    1
2024-07    1
2024-06    1
2024-05    1
2024-04    1
2024-03    1
2024-02    1
2024-01    1
2023-12    1
Name: count, dtype: int64


In [ ]:
# uses county 25007, or dukes county, to fill in the nantucket values
# .set_index("date") creates a smaller df where indices are the dates
donor = df[df["GEOID"] == "25007"].set_index("date")

# mask is a df of True/False values. for every row in df, if its 25019: true; else: false
mask = df["GEOID"]=="25019"

# left side: looks for the true values in the mask df. if true, it specifically highlights the soil moisture column
# right side: for every true value in df, it finds the date and uses .map() to search for it in the donor
  # since donor set the index to date, this allows you to directly look up that specific row, grab the soil moisture and assign it
df.loc[mask, "volumetric_soil_water_layer_3"] = df.loc[mask, "date"].map(donor["volumetric_soil_water_layer_3"])
df.loc[mask, "total_precipitation_sum"] = df.loc[mask, "date"].map(donor["total_precipitation_sum"])

In [ ]:
# no more NA rows
df["total_precipitation_sum"].isna().sum()

np.int64(0)

In [ ]:
df = df.sort_values(["GEOID", "date"])

# shifts the data by 1 and then sums the past 3 months
# example: if you're in april and apply shift(1), april now holds march's value. rolling(3).sum() then gets march, feb, and jan values and sums
df["precip_prior3"] = (
    df.groupby("GEOID")["total_precipitation_sum"].transform(lambda s: s.shift(1).rolling(3).sum())
)

df["precip_prior6"] = (
    df.groupby("GEOID")["total_precipitation_sum"].transform(lambda s: s.shift(1).rolling(6).sum())
)

In [ ]:
df

,GEOID,date,volumetric_soil_water_layer_3,total_precipitation_sum,precip_prior3,precip_prior6
56,01001,1950-02,0.442949,0.080213,NaN,NaN
3164,01001,1950-03,0.459730,0.137679,NaN,NaN
6272,01001,1950-04,0.442017,0.073473,NaN,NaN
9380,01001,1950-05,0.413145,0.093772,0.291365,NaN
12488,01001,1950-06,0.372161,0.053034,0.304924,NaN
...,...,...,...,...,...,...
2837240,56045,2026-02,0.192497,0.008521,0.044310,0.141030
2840348,56045,2026-03,0.192373,0.013870,0.036165,0.123029
2843456,56045,2026-04,0.190505,0.037310,0.031929,0.121160
2846564,56045,2026-05,0.187651,0.028983,0.059701,0.104011


In [ ]:
df = df.drop(columns="total_precipitation_sum")
df

,GEOID,date,volumetric_soil_water_layer_3,precip_prior3,precip_prior6
56,01001,1950-02,0.442949,NaN,NaN
3164,01001,1950-03,0.459730,NaN,NaN
6272,01001,1950-04,0.442017,NaN,NaN
9380,01001,1950-05,0.413145,0.291365,NaN
12488,01001,1950-06,0.372161,0.304924,NaN
...,...,...,...,...,...
2837240,56045,2026-02,0.192497,0.044310,0.141030
2840348,56045,2026-03,0.192373,0.036165,0.123029
2843456,56045,2026-04,0.190505,0.031929,0.121160
2846564,56045,2026-05,0.187651,0.059701,0.104011


In [ ]:
df.to_parquet("/content/drive/MyDrive/Nino/cleaned_data/precip_soil_dynamic.parquet")